# Stage 3 - Apply Taxonomy and Rewrite Titles

This notebook reads the raw Markdown exports from stage 1, applies the taxonomy from stage 2, generates a better note title, and writes enriched copies into a separate output location. It never overwrites the raw export folder.

## Workflow

Inputs:
- `DeepSeek_Exports/*.md` from the export notebook
- `taxonomy_state/taxonomy_v1.json` from the taxonomy notebook

Outputs:
- `DeepSeek_Enriched/*.md` as the local intermediate layer
- optional copy into a separate Obsidian vault path via `ENRICHED_OBSIDIAN_VAULT_PATH`
- `DeepSeek_Enriched/enrichment_manifest.json` for incremental reruns

Recommended order:
1. Run the configuration and helper cells
2. Run the preview cell on one note
3. Run the batch cell when the preview looks right

In [ ]:
from __future__ import annotations

import json
import os
import re
import hashlib
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests

In [ ]:
PROJECT_ROOT = Path.cwd()
SOURCE_EXPORT_DIR = PROJECT_ROOT / "DeepSeek_Exports"
STATE_DIR = PROJECT_ROOT / "taxonomy_state"
TAXONOMY_FILE = STATE_DIR / "taxonomy_v1.json"

ENRICHED_DIR = PROJECT_ROOT / "DeepSeek_Enriched"
ENRICHED_DIR.mkdir(exist_ok=True)
MANIFEST_FILE = ENRICHED_DIR / "enrichment_manifest.json"

target_vault_raw = os.getenv("ENRICHED_OBSIDIAN_VAULT_PATH", "").strip()
ENRICHED_OBSIDIAN_VAULT_PATH = Path(target_vault_raw) if target_vault_raw else None
ENRICHED_OBSIDIAN_SUBFOLDER = os.getenv("ENRICHED_OBSIDIAN_SUBFOLDER", "DeepSeek Tagged")
ENRICHED_OBSIDIAN_DIR = (ENRICHED_OBSIDIAN_VAULT_PATH / ENRICHED_OBSIDIAN_SUBFOLDER) if ENRICHED_OBSIDIAN_VAULT_PATH else None
if ENRICHED_OBSIDIAN_DIR:
    ENRICHED_OBSIDIAN_DIR.mkdir(parents=True, exist_ok=True)

PROVIDER = os.getenv("TAXONOMY_PROVIDER", "ollama")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "")

DEFAULT_TAG_COUNT = 3
MAX_BODY_CHARS = int(os.getenv("ENRICH_MAX_BODY_CHARS", "50000"))

print({
    "source_export_dir": str(SOURCE_EXPORT_DIR),
    "taxonomy_file": str(TAXONOMY_FILE),
    "enriched_dir": str(ENRICHED_DIR),
    "obsidian_target": str(ENRICHED_OBSIDIAN_DIR) if ENRICHED_OBSIDIAN_DIR else None,
    "provider": PROVIDER,
    "model": OLLAMA_MODEL if PROVIDER == "ollama" else OPENAI_MODEL,
})

In [ ]:
def extract_frontmatter_and_body(text: str) -> tuple[dict[str, str], str]:
    if text.startswith("---\n"):
        parts = text.split("---\n", 2)
        if len(parts) == 3:
            fm_raw, body = parts[1], parts[2]
            fm: dict[str, str] = {}
            for line in fm_raw.splitlines():
                if ":" in line:
                    key, value = line.split(":", 1)
                    fm[key.strip()] = value.strip().strip('"')
            return fm, body.strip()
    return {}, text.strip()


def detect_language_simple(text: str) -> str:
    zh_chars = len(re.findall(r"[\u4e00-\u9fff]", text))
    en_chars = len(re.findall(r"[A-Za-z]", text))
    total = zh_chars + en_chars
    if total == 0:
        return "unknown"
    zh_ratio = zh_chars / total
    if zh_ratio > 0.65:
        return "zh"
    if zh_ratio < 0.35:
        return "en"
    return "mixed"


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:16]


def sanitize_filename(name: str) -> str:
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r'[_\s]+', ' ', name).strip()
    return name[:100].strip() or "Untitled"


def make_filename(title: str, date: str) -> str:
    return f"{date} {sanitize_filename(title)}.md"


def load_manifest() -> dict[str, Any]:
    if MANIFEST_FILE.exists():
        return json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
    return {"last_updated": None, "notes": []}


def save_manifest(manifest: dict[str, Any]) -> None:
    manifest["last_updated"] = datetime.now(timezone.utc).isoformat()
    MANIFEST_FILE.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def manifest_index(manifest: dict[str, Any]) -> dict[str, dict[str, Any]]:
    return {entry["source_key"]: entry for entry in manifest.get("notes", [])}


def frontmatter_list(key: str, values: list[str]) -> list[str]:
    lines = [f"{key}:"]
    lines.extend([f"  - {value}" for value in values])
    return lines


def render_enriched_markdown(note: dict[str, Any], decision: dict[str, Any]) -> str:
    refined_title = decision["refined_title"].replace('"', '\\"')
    original_title = note["original_title"].replace('"', '\\"')
    lines = [
        "---",
        f'date: "{note["date"]}"',
        f'original_title: "{original_title}"',
        f'refined_title: "{refined_title}"',
        f'conversation_id: "{note["conversation_id"]}"',
        f'url: "{note["url"]}"',
    ]
    lines.extend(frontmatter_list("tags", decision["tag_labels"]))
    lines.extend(frontmatter_list("taxonomy_category_ids", decision["tag_category_ids"]))
    lines.extend([
        f'classification_confidence: {decision["confidence"]}',
        "---",
        "",
        f'# {decision["refined_title"]}',
        "",
        f'_Original title: {note["original_title"]}_',
        "",
        "## Applied Taxonomy",
        "",
        f'- Tags: {", ".join(decision["tag_labels"])}',
        f'- Category IDs: {", ".join(decision["tag_category_ids"])}',
        f'- Confidence: {decision["confidence"]}',
        f'- Reason: {decision["reason"]}',
        "",
        "## Conversation",
        "",
        note["body"],
    ])
    return "\n".join(lines).strip() + "\n"


def load_source_notes() -> list[dict[str, Any]]:
    if not SOURCE_EXPORT_DIR.exists():
        raise FileNotFoundError(f"Source export folder not found: {SOURCE_EXPORT_DIR}")

    notes = []
    for path in sorted(SOURCE_EXPORT_DIR.glob("*.md")):
        raw = path.read_text(encoding="utf-8", errors="ignore")
        frontmatter, body = extract_frontmatter_and_body(raw)
        original_title = frontmatter.get("original_title") or frontmatter.get("title") or path.stem
        conversation_id = frontmatter.get("conversation_id") or path.stem
        date = frontmatter.get("date", path.stem.split(" ", 1)[0])
        url = frontmatter.get("url", "")
        notes.append({
            "source_key": conversation_id,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "filename": path.name,
            "conversation_id": conversation_id,
            "date": date,
            "url": url,
            "original_title": original_title,
            "body": body,
            "language": detect_language_simple(body),
            "content_hash": stable_hash(raw),
        })
    return notes

In [ ]:
def llm_call_json(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> dict[str, Any]:
    if PROVIDER == "ollama":
        payload = {
            "model": OLLAMA_MODEL,
            "prompt": f"<|system|>\n{system_prompt}\n<|user|>\n{user_prompt}\n<|assistant|>",
            "stream": False,
            "options": {"temperature": temperature},
            "format": "json",
        }
        response = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=300)
        response.raise_for_status()
        text = response.json().get("response", "{}")
        return json.loads(text)

    if PROVIDER == "openai_compatible":
        if not OPENAI_BASE_URL or not OPENAI_API_KEY or not OPENAI_MODEL:
            raise ValueError("Set OPENAI_BASE_URL, OPENAI_API_KEY, and OPENAI_MODEL first")

        url = OPENAI_BASE_URL.rstrip("/") + "/chat/completions"
        headers = {
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": OPENAI_MODEL,
            "temperature": temperature,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        }
        response = requests.post(url, headers=headers, json=payload, timeout=300)
        response.raise_for_status()
        text = response.json()["choices"][0]["message"]["content"]
        return json.loads(text)

    raise ValueError(f"Unsupported provider: {PROVIDER}")


def load_taxonomy() -> dict[str, Any]:
    if not TAXONOMY_FILE.exists():
        raise FileNotFoundError(f"Taxonomy file not found: {TAXONOMY_FILE}")

    taxonomy = json.loads(TAXONOMY_FILE.read_text(encoding="utf-8"))
    categories = taxonomy.get("categories", [])
    if not categories:
        raise ValueError("Taxonomy file has no categories")

    taxonomy["category_ids"] = [category["category_id"] for category in categories]
    taxonomy["category_lookup"] = {category["category_id"]: category for category in categories}
    taxonomy["taxonomy_hash"] = stable_hash(json.dumps(categories, ensure_ascii=False, sort_keys=True))
    return taxonomy


def enrich_one_note(note: dict[str, Any], taxonomy: dict[str, Any], tag_count: int = DEFAULT_TAG_COUNT) -> dict[str, Any]:
    categories = taxonomy["categories"]
    category_ids = taxonomy["category_ids"]
    category_lookup = taxonomy["category_lookup"]

    system_prompt = (
        "You rewrite conversation note titles and assign taxonomy tags. Return strict JSON only. "
        "Use only category_id values from the provided taxonomy. "
        "Prefer 2 to 4 tags, avoid generic titles, and keep the refined title concise but descriptive."
    )

    user_prompt = json.dumps({
        "task": "Classify one exported DeepSeek conversation and rewrite its title",
        "tag_count_target": tag_count,
        "rules": [
            "Read the conversation history before choosing tags",
            "Choose only category_id values from the taxonomy",
            "Return 2 to 4 tags unless the content is truly unclear",
            "The refined title should help a future reader find the note quickly",
            "Do not invent topics that are not supported by the conversation text",
        ],
        "taxonomy": categories,
        "document": {
            "original_title": note["original_title"],
            "language": note["language"],
            "conversation_markdown": note["body"][:MAX_BODY_CHARS],
        },
        "output_schema": {
            "refined_title": "string",
            "tag_category_ids": ["enum"],
            "confidence": "0..1",
            "reason": "short string",
        },
    }, ensure_ascii=False)

    raw = llm_call_json(system_prompt, user_prompt, temperature=0.0)

    refined_title = str(raw.get("refined_title") or note["original_title"]).strip()
    raw_ids = raw.get("tag_category_ids") or []
    if not isinstance(raw_ids, list):
        raw_ids = []

    tag_category_ids = []
    for category_id in raw_ids:
        category_id = str(category_id).strip()
        if category_id in category_ids and category_id not in tag_category_ids:
            tag_category_ids.append(category_id)

    fallback_id = "unclear" if "unclear" in category_ids else ("other" if "other" in category_ids else category_ids[0])
    if not tag_category_ids:
        tag_category_ids = [fallback_id]
    tag_category_ids = tag_category_ids[:4]

    confidence = raw.get("confidence", 0.0)
    if not isinstance(confidence, (int, float)):
        confidence = 0.0

    reason = str(raw.get("reason", "")).strip()[:280]
    tag_labels = [category_lookup[category_id]["label"] for category_id in tag_category_ids]

    return {
        "refined_title": refined_title,
        "tag_category_ids": tag_category_ids,
        "tag_labels": tag_labels,
        "confidence": round(float(confidence), 4),
        "reason": reason,
    }

In [ ]:
def write_enriched_note(note: dict[str, Any], decision: dict[str, Any], taxonomy_hash: str) -> dict[str, Any]:
    rendered = render_enriched_markdown(note, decision)
    filename = make_filename(decision["refined_title"], note["date"])

    local_path = ENRICHED_DIR / filename
    local_path.write_text(rendered, encoding="utf-8")

    obsidian_path = None
    if ENRICHED_OBSIDIAN_DIR is not None:
        obsidian_path = ENRICHED_OBSIDIAN_DIR / filename
        obsidian_path.write_text(rendered, encoding="utf-8")

    return {
        "source_key": note["source_key"],
        "conversation_id": note["conversation_id"],
        "source_file": note["filename"],
        "output_file": filename,
        "local_output_path": str(local_path.relative_to(PROJECT_ROOT)),
        "obsidian_output_path": str(obsidian_path) if obsidian_path else None,
        "refined_title": decision["refined_title"],
        "tags": decision["tag_labels"],
        "taxonomy_category_ids": decision["tag_category_ids"],
        "confidence": decision["confidence"],
        "reason": decision["reason"],
        "source_hash": note["content_hash"],
        "taxonomy_hash": taxonomy_hash,
        "signature": stable_hash(note["content_hash"] + "::" + taxonomy_hash),
        "updated_at": datetime.now(timezone.utc).isoformat(),
    }


def preview_one_note(md_path: str | None = None, tag_count: int = DEFAULT_TAG_COUNT) -> dict[str, Any]:
    taxonomy = load_taxonomy()
    notes = load_source_notes()
    if not notes:
        raise FileNotFoundError(f"No markdown files found in {SOURCE_EXPORT_DIR}")

    if md_path is None:
        note = notes[0]
    else:
        wanted = md_path.replace("\\", "/")
        note = next((item for item in notes if item["path"] == wanted or item["filename"] == Path(wanted).name), None)
        if note is None:
            raise FileNotFoundError(f"Could not find markdown note: {md_path}")

    decision = enrich_one_note(note, taxonomy, tag_count=tag_count)
    return {
        "path": note["path"],
        "original_title": note["original_title"],
        "refined_title": decision["refined_title"],
        "tags": decision["tag_labels"],
        "taxonomy_category_ids": decision["tag_category_ids"],
        "confidence": decision["confidence"],
        "reason": decision["reason"],
    }


def enrich_all_notes(limit: int | None = None, force: bool = False, tag_count: int = DEFAULT_TAG_COUNT) -> list[dict[str, Any]]:
    taxonomy = load_taxonomy()
    notes = load_source_notes()
    manifest = load_manifest()
    existing = manifest_index(manifest)

    selected_notes = notes if limit is None else notes[:limit]
    results = []
    refreshed_entries = []

    for note in selected_notes:
        signature = stable_hash(note["content_hash"] + "::" + taxonomy["taxonomy_hash"])
        prior = existing.get(note["source_key"])
        if not force and prior and prior.get("signature") == signature:
            results.append({
                "status": "skipped",
                "source_file": note["filename"],
                "refined_title": prior.get("refined_title"),
                "tags": prior.get("tags", []),
            })
            refreshed_entries.append(prior)
            continue

        decision = enrich_one_note(note, taxonomy, tag_count=tag_count)
        entry = write_enriched_note(note, decision, taxonomy["taxonomy_hash"])
        refreshed_entries.append(entry)
        results.append({
            "status": "updated",
            "source_file": note["filename"],
            "output_file": entry["output_file"],
            "refined_title": entry["refined_title"],
            "tags": entry["tags"],
        })

    untouched_keys = {note["source_key"] for note in selected_notes}
    for old_entry in manifest.get("notes", []):
        if old_entry.get("source_key") not in untouched_keys:
            refreshed_entries.append(old_entry)

    manifest["notes"] = sorted(refreshed_entries, key=lambda item: item.get("source_file", ""))
    save_manifest(manifest)
    return results

In [ ]:
# Preview one note before running the full batch.
PREVIEW_MD_PATH = None
# Example: PREVIEW_MD_PATH = "DeepSeek_Exports/2026-04-20 Some Title.md"

preview = preview_one_note(PREVIEW_MD_PATH)
print(json.dumps(preview, ensure_ascii=False, indent=2))

In [ ]:
# Run the full enrichment pipeline. Set FORCE_RERUN = True to rebuild all notes.
LIMIT = None
FORCE_RERUN = False

results = enrich_all_notes(limit=LIMIT, force=FORCE_RERUN)
updated = [row for row in results if row["status"] == "updated"]
skipped = [row for row in results if row["status"] == "skipped"]
print(json.dumps({
    "total": len(results),
    "updated": len(updated),
    "skipped": len(skipped),
    "sample_updated": updated[:3],
}, ensure_ascii=False, indent=2))

## Notes

- If `taxonomy_state/taxonomy_v1.json` does not exist yet, run the taxonomy notebook first.
- If `ENRICHED_OBSIDIAN_VAULT_PATH` is unset, the notebook still writes the local intermediate files into `DeepSeek_Enriched/`.
- The manifest uses a combined source-content hash and taxonomy hash, so reruns skip notes that have not changed.